# 04.2 · kv-transfer (Spec-2) — data pipeline demo

`SRC.task != DST.task`, `SRC.progress == DST.progress`. Two distinct tasks that share a **common progress** (a prerequisite sub-computation) are clustered together; both prefill the same progress code. Transplant the SRC **task-description key+value** onto DST (GQA-corrected headwise routing) → does DST implement SRC's task?

Also demonstrates **cross-language** transfer: cache the task in Python, trigger it in C++.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-10-16 19:29:06 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
import json
with open(
    os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR), "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")
) as f:
    GOAL_HEADS_250 = [(layer_idx, head_idx) for layer_idx, head_idx in json.load(f)]

## Intervention knobs

`ALPHA` — OV-write boost of `PATCH_HEADS` at the pivot during the **patched** gen/readout (`1.0` = off) — the only knob; every intervention here is a **span** k+v transplant (no pivot-query window). Default is a no-op.

In [3]:
from contextlib import nullcontext

#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
QUERY_POSITIONS = "last"  # which DST query positions of PATCH_HEADS read the transplanted span:
                         #   "all"  = every query after the span (original; pre-pivot latents change too)
                         #   "last" = only the last query of each forward (pivot, then each generated token)
                         #            -> pre-pivot DST latents are untouched
                         #   "last_n" = the last QUERY_LAST_N prompt tokens (ending at the pivot), then each
                         #            generated token -> latents before them are untouched
QUERY_LAST_N = 4         # only used when QUERY_POSITIONS == "last_n" (1 == "last")
QP = QUERY_LAST_N if QUERY_POSITIONS == "last_n" else QUERY_POSITIONS   # what headwise_kv_patch receives
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
READOUT_HEADS = GOAL_HEADS                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 43)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------

def ov_boost(heads=None, alpha=None, positions=(-1,)):
    """Context manager scaling the OV write of `heads` by `alpha` (defaults PATCH_HEADS/ALPHA);
    alpha==1.0 -> nullcontext (no-op). Stack it under `headwise_kv_patch` for the patched gen."""
    heads = PATCH_HEADS if heads is None else heads
    alpha = ALPHA if alpha is None else alpha
    return boost_head_ov(mt, heads, alpha, positions=positions) if alpha != 1.0 else nullcontext()



def ablate_ctx(heads=None, keep=None, block_span=None):
    """Ablate `heads` (default ABLATE_HEADS) on the PATCHED run; `keep` heads (default PATCH_HEADS)
    are never ablated so a patched head is never touched. Strategy per ABLATION_STRAT:
      "zero"     -> kill their OV write (boost_head_ov x0.0 at ALL positions);
      "restrict" -> attention-knockout: block their attention to `block_span` (the DST <task_span>)
                    so they can look everywhere else. "restrict" requires `block_span=(start, end)`."""
    heads = ABLATE_HEADS if heads is None else heads
    keep = set(PATCH_HEADS if keep is None else keep)
    heads = [h for h in heads if h not in keep]
    if not heads:
        return nullcontext()
    if ABLATION_STRAT == "restrict":
        assert block_span is not None, "restrict ablation needs block_span (the DST task span)"
        return restrict_head_attn(mt, heads, range(block_span[0], block_span[1]), keep=keep)
    return boost_head_ov(mt, heads, 0.0, positions=None)

def headwise_iface(patch_heads, k_cache, v_cache, dst_pos, alpha=None, query_positions=None):
    """headwise_kv_patch patch_interface, composed with the ALPHA OV-boost — for the PATCHED
    goalscope readout. At ALPHA==1.0 it matches the plain `make_headwise_interface` iface.
    `query_positions` defaults to QP (read at call time)."""
    def _if(mt, inputs, patches=None, model_kwargs=None):
        qp = QP if query_positions is None else query_positions
        with headwise_kv_patch(mt, patch_heads, k_cache, v_cache, dst_pos, query_positions=qp), ov_boost(patch_heads, alpha):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


# ── two-span prompt renderer: full task span (bright_yellow) + the EXACT KV window (orange) ────
# The transplant window is chosen by `align_kv_spans(mt, ..., mode=ALIGN_KNOB)` (src.diff.functional).
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Render the prompt exactly as the model sees it. task_span -> bright_yellow (the task text);
    kv_span -> `black on dark_orange` (the exact tokens whose k+v are transplanted); pivot -> salmon.
    Priority pivot > kv > task, so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids)
    pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    # Mark spans with control-char sentinels and escape the WHOLE joined string once (then swap
    # sentinels for tags). Escaping per-token drops brackets like `[w ...]` / `list[str]` when the
    # tokenizer splits `[` from the following letter across two tokens: the lone `[` isn't escaped,
    # so once joined it reads as a markup tag and rich absorbs it.
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Inspect the generated clusters / setups

In [4]:
for dt, clusters in D.KVCLUSTERS.items():
    print(f"{dt}:")
    for cl in clusters:
        print(f"  cluster {cl.id!r}: prereq={cl.common_progress.result_name!r} "
              f"({cl.common_progress.description!r}) · tasks={[t.id for t in cl.tasks]}")
print()
setups = D.kv_transfer_setups("list[str]", presentation="description")
print(f"list[str]: {len(setups)} ordered pairs across its clusters")
for s in setups[:12]:
    print(f"  [{s.meta['cluster']:<14}] src={s.src_task:<22} dst={s.dst_task:<22} pivot={s.pivot}")

list[str]:
  cluster 'sorted_by_len': prereq='sorted_words' ('the words sorted by length') · tasks=['shortest_vowel_start', 'longest_palindrome', 'longest_len', 'shortest_len', 'median_word_len', 'longest_starts_upper']
  cluster 'long_words': prereq='long_words' ('the words that have at least two characters') · tasks=['count_palindromes', 'count_all_distinct', 'count_same_ends']
  cluster 'lowercased': prereq='lowered' ('the words converted to lowercase') · tasks=['count_ci_palindromes', 'num_distinct_ci', 'count_contains_a', 'any_ci_duplicate']
list[int]:
  cluster 'sorted_asc': prereq='sorted_nums' ('the values sorted in ascending order') · tasks=['median_val', 'range_span', 'second_largest', 'second_smallest', 'max_gap']
  cluster 'positives': prereq='positives' ('the values that are greater than zero') · tasks=['sum_positives', 'count_positives', 'product_positives', 'smallest_positive', 'all_even_positives']
  cluster 'prefix_sums': prereq='prefix' ('the running cumulative sums o

## Render one setup — SRC/DST prompts with the transplanted task span marked

In [5]:
inst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="cpp", dst_lang="python",
)[4]
m = D.materialize(inst)
SRC_SPAN0 = token_span(m["src_prompt"], m["src_task_text"])   # located target spans (pre-align)
DST_SPAN0 = token_span(m["dst_prompt"], m["dst_task_text"])
print(f"SRC task = {m['src_task_text']!r}\nDST task = {m['dst_task_text']!r}")
# align -> TokenizerOutputs (pad token-splices \n before the shorter target). SRC_POS = cached-from,
# DST_POS = patched-to. For pad both spans are K=max -> the whole span is orange (fully patched).
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], SRC_SPAN0, DST_SPAN0, mode=ALIGN_KNOB,
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
piv_src, piv_dst = src_tok["input_ids"].shape[1] - 1, dst_tok["input_ids"].shape[1] - 1
show_spans(src_tok, f"SRC · CACHED-FROM {tuple(SRC_SPAN)} (orange) · pivot@{piv_src} · align={ALIGN_KNOB}",
           task_span=SRC_SPAN, kv_span=(SRC_POS[0], SRC_POS[-1] + 1), piv=-1)
show_spans(dst_tok, f"DST · PATCHED-TO {tuple(DST_SPAN)} (orange) · pivot@{piv_dst} · align={ALIGN_KNOB}",
           task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), piv=-1)

SRC task = 'the length of the median word when the words are ordered by length'
DST task = 'the length of the shortest word'


╭─ SRC · CACHED-FROM (64, 77) (orange) · pivot@158 · align=pad ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ the length of the median word when the words are ordered by length\n                                            │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::vector<std::string> sorted_words = words;\n                                                            │
│     std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){  │
│ return a.size() < b.size(); });\n                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · PATCHED-TO (63, 76) (orange) · pivot@103 · align=pad ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ the length of the shortest word\n                                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant SRC task-description key+value onto DST

Align the two task spans (per `ALIGN_KNOB` — right-edge or pad), cache SRC key+value over `PATCH_HEADS`, and run the destination under `headwise_kv_patch` (only `PATCH_HEADS` read the source span — GQA bleed corrected). Expect patched DST to implement SRC's task.

In [6]:
# ── intervention: transplant SRC task-description k+v onto DST — grouped per run ──
from src.diff.functional import _layers_of, heads_per_layer, cache_src_proj

# ALPHA = 3.0
# PATCH_HEADS = UNION_HEADS

# src_tok / dst_tok / SRC_POS (cached-from) / DST_POS (patched-to) / piv come from the render cell above
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "k")
v_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "v")

# ── source_run — clean SRC ──
gen_src = greedy(src_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC ({m['src_task_text']!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_tok, piv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST ──
gen_dst = greedy(dst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST ({m['dst_task_text']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_tok, piv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — SRC task k+v transplanted (+ ALPHA OV-boost) · QUERY_POSITIONS: all / last queries ──
_boost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS, query_positions=QP), _boost, ablate_ctx(keep=PATCH_HEADS, block_span=DST_SPAN):
    _patched = greedy(dst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {m['src_task_text']!r}):[/salmon1] \n{escape(_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_tok, piv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, k_cache, v_cache, DST_POS))[0]["patched_gen"]

# ── eval pipeline — causal + readout (no gaze) ──
from src.diff.eval import eval_kv_transfer
_ev = eval_kv_transfer(inst, patched_gen=_patched, readouts=(ro_src, ro_dst, ro_patched))
rich_print(f"[bold]eval · kv-cluster[/bold] : {_ev.summary()}")

source_run · clean SRC ('the length of the median word when the words are ordered by length'): 
 size_t median_index = sorted_words.size() / 2;
     return sorted_words[median_index].size();
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=158 '    ' => I'm identifying the middle element of a sorted list or array.

destination_run · clean DST ('the length of the shortest word'): 
 len(sorted_words[0])
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=103 ' return' => I'm determining the length of the shortest string in the 
list by using the `len()` function on the first element after sorting the list by string length.

patched_run · DST (expect 'the length of the median word when the words are ordered by length'): 
 len(sorted_words[0]) if sorted_words else 0
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=103 ' return' => I'm calculating the length of the shortest string in a list by 
sorting the list by string length and returning the length of the first element.

eval · kv-cluster : causal: no (dst) | readout: src→dst→dst

### Check — do the pre-pivot DST latents change under the transplant?

Residual stream (every layer output) of the clean DST run vs the patched run (k+v transplant + ALPHA boost, no
ablation): max |Δ| over the pre-pivot positions after the task span, and at the pivot. `"last"` / an int n should give exactly 0
before the routed queries (the pivot / the last n prompt tokens); `"all"` should not (there every pre-pivot
position after the span is routed).

In [7]:
import baukit
from contextlib import ExitStack


def residuals(tok, ctx_fn=nullcontext):
    names = [mt.layer_name_format.format(l) for l in range(mt.n_layer)]
    with torch.no_grad(), ctx_fn(), baukit.TraceDict(mt._model, names, retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    out = lambda o: (o[0] if isinstance(o, tuple) else o)
    return torch.stack([out(tr[n].output)[0].float().cpu() for n in names])   # (L, T, d)


def patched_ctx(qp):
    def _c():
        s = ExitStack()
        s.enter_context(headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS, query_positions=qp))
        s.enter_context(ov_boost(PATCH_HEADS, ALPHA))
        return s
    return _c


H_dst = residuals(dst_tok)
span_end, piv = DST_SPAN[1], dst_tok["input_ids"].shape[1] - 1
for qp in dict.fromkeys(("all", "last", QP)):  # + the knob's value
    d = (residuals(dst_tok, patched_ctx(qp)) - H_dst).abs()                    # (L, T, d)
    lo = piv if qp == "all" else piv - (1 if qp == "last" else qp) + 1          # first routed query (all: n/a)
    end = piv if qp == "all" else lo                                            # region that SHOULD be untouched
    pre = d[:, span_end:end].amax(dim=(1, 2)) if end > span_end else torch.zeros(1)
    at_piv = d[:, piv].amax(dim=-1)
    first = int((pre > 0).nonzero()[0]) if (pre > 0).any() else None
    rich_print(f"[bold]{str(qp):>4}[/bold] · max|Δ| after span, before the routed queries ({end - span_end} pos) = "
               f"{pre.max():.4g} (first layer ≠ 0: {first}) · at pivot = {at_piv.max():.4g}")

 all · max|Δ| after span, before the routed queries (27 pos) = 360 (first layer ≠ 0: 28) · at pivot = 61

last · max|Δ| after span, before the routed queries (27 pos) = 0 (first layer ≠ 0: None) · at pivot = 112

## Cross-language — cache the task in Python, trigger it in a C++ destination

In [8]:
# ── cross-language: cache the task in Python, trigger it in a C++ destination — grouped per run ──
xinst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="cpp", dst_lang="python"
)[3]
xm = D.materialize(xinst)
XS0 = token_span(xm["src_prompt"], xm["src_task_text"])
XD0 = token_span(xm["dst_prompt"], xm["dst_task_text"])
xsrc_tok, xdst_tok, XSP, XDP, XS, XD = align_kv_spans(
    mt, xm["src_prompt"], xm["dst_prompt"], XS0, XD0, mode=ALIGN_KNOB,
    src_text=xm["src_task_text"], dst_text=xm["dst_task_text"])
xpiv_src, xpiv_dst = xsrc_tok["input_ids"].shape[1] - 1, xdst_tok["input_ids"].shape[1] - 1
show_spans(xsrc_tok, f"SRC (python) · CACHED-FROM {tuple(XS)} (orange) · align={ALIGN_KNOB}",
           task_span=XS, kv_span=(XSP[0], XSP[-1] + 1), piv=-1)
show_spans(xdst_tok, f"DST (c++)    · PATCHED-TO {tuple(XD)} (orange) · align={ALIGN_KNOB}",
           task_span=XD, kv_span=(XDP[0], XDP[-1] + 1), piv=-1)

xkvh = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
xk = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "k")
xv = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "v")

# ── source_run (python) ──
xgen_src = greedy(xsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC/python ({xm['src_task_text']!r}):[/red1] \n{escape(xgen_src)}")
xro_src = goalscope_readout("   ↳ goalscope · source_run", xsrc_tok, xpiv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run (c++) ──
xgen_dst = greedy(xdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST/c++ ({xm['dst_task_text']!r}):[/sea_green3] \n{escape(xgen_dst)}")
xro_dst = goalscope_readout("   ↳ goalscope · destination_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — python task k+v -> c++ DST (expect the PYTHON src task) ──
_xboost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, xk, xv, XDP, query_positions=QP), _xboost, ablate_ctx(keep=PATCH_HEADS, block_span=XD):
    xpatched = greedy(xdst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST/c++ (expect {xm['src_task_text']!r}):[/salmon1] \n{escape(xpatched)}")
xro_patched = goalscope_readout("   ↳ goalscope · patched_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, xk, xv, XDP))[0]["patched_gen"]

from src.diff.eval import eval_kv_transfer
_xev = eval_kv_transfer(xinst, patched_gen=xpatched, readouts=(xro_src, xro_dst, xro_patched))
rich_print(f"[bold]eval · kv-cluster (cross-lang)[/bold] : {_xev.summary()}")

╭─ SRC (python) · CACHED-FROM (64, 74) (orange) · align=pad ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ returns true if the shortest word starts with a vowel\n                                                         │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::vector<std::string> sorted_words = words;\n                                                            │
│     std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){  │
│ return a.size() < b.size(); });\n                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST (c++)    · PATCHED-TO (63, 73) (orange) · align=pad ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ returns true if the longest word is a palindrome\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

source_run · clean SRC/python ('returns true if the shortest word starts with a vowel'): 
 char first_char = std::tolower(sorted_words[0][0]);
    return first_char == 'a' || first_char == 'e' || first_char == 'i' || first_char == 'o' || first_char == 'u';
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=155 '    ' => I'm checking if the first character of the string is a vowel.

destination_run · clean DST/c++ ('returns true if the longest word is a palindrome'): 
 sorted_words[-1] == sorted_words[-1][::-1]
</code>

... (2 outputs left out)

patched_run · DST/c++ (expect 'returns true if the shortest word starts with a vowel'): 
 sorted_words[0][0].lower() in 'aeiou'
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=100 ' return' => I'm checking if the first character of the string is a vowel.

eval · kv-cluster (cross-lang) : causal: TRIGGERED (src) | readout: src→dst→src

In [9]:
print(mt.tokenizer.decode(src_tok.input_ids[0].tolist()))

<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
the length of the median word when the words are ordered by length
Assume that all the libraries have been already imported.
<|im_end|>
<|im_start|>assistant
<code>
auto func(std::vector<std::string> words){
    std::vector<std::string> sorted_words = words;
    std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){ return a.size() < b.size(); });
    


## Multi-predicate setup — `SRC(T1,T2,T3)` vs `DST(T1,T2,T4)` (kv-transfer)

A kv-transfer variant purpose-built for this notebook. SRC and DST share a **prefilled prefix** (T1, T2 assigned identically) and diverge only in the **frontier** predicate their *question* lists — SRC asks for T3, DST for T4. The frontier description is a distinct span (T3 in SRC's question, T4 in DST's), so this is a genuine **k+v transplant** (not query): cache the SRC **T3-description** key+value and transplant it onto the DST **T4-description** span. Test: does patched DST implement **T3** instead of T4? Gaze is **N/A** (T3 never appears in the DST prompt); eval = causal + readout.

In [10]:
ALIGN_KNOB = "pad"
# PATCH_HEADS = UNION_HEADS
ALPHA = 2.0
PATCH_HEADS = [(layer_idx, head_idx) for layer_idx in range(35, 50) for head_idx in range(mt.n_heads)]
len(PATCH_HEADS)

600

In [11]:
mp = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=0,
    src_lang="cpp", dst_lang="python", naming="letters", letter_stub="var_eq"
)
for s in mp:
    print(f"  shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<16} DST.frontier={s.dst_next}")

inst_mp = mp[7]
mm = D.materialize(inst_mp)
MP_SRC0 = token_span(mm["src_prompt"], mm["src_task_text"])   # T3 description (pre-align)
MP_DST0 = token_span(mm["dst_prompt"], mm["dst_task_text"])   # T4 description (pre-align)
print(f"\nSRC frontier (T3) = {mm['src_task_text']!r}")
print(f"DST frontier (T4) = {mm['dst_task_text']!r}")
print(f"shared (T1,T2)    = {inst_mp.task_desc!r}")
mp_src_tok, mp_dst_tok, MP_SRC_POS, MP_DST_POS, MP_SRC_SPAN, MP_DST_SPAN = align_kv_spans(
    mt, mm["src_prompt"], mm["dst_prompt"], MP_SRC0, MP_DST0, mode=ALIGN_KNOB,
    src_text=mm["src_task_text"], dst_text=mm["dst_task_text"])
show_spans(mp_src_tok, f"SRC · T3 CACHED-FROM {tuple(MP_SRC_SPAN)} (orange) · pivot@{mp_src_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_SRC_SPAN, kv_span=(MP_SRC_POS[0], MP_SRC_POS[-1] + 1), piv=-1)
show_spans(mp_dst_tok, f"DST · T4 PATCHED-TO {tuple(MP_DST_SPAN)} (orange) · pivot@{mp_dst_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_DST_SPAN, kv_span=(MP_DST_POS[0], MP_DST_POS[-1] + 1), piv=-1)

  shared=['all_unique_lengths', 'has_palindrome']  SRC.frontier=distinct_count   DST.frontier=count_short
  shared=['any_empty', 'count_ends_consonant']  SRC.frontier=total_uppercase_letters DST.frontier=distinct_count
  shared=['total_vowels', 'longest_word_length']  SRC.frontier=count_capitalized DST.frontier=count_short
  shared=['longest_word_length', 'count_long']  SRC.frontier=any_empty        DST.frontier=total_length
  shared=['distinct_count', 'total_uppercase_letters']  SRC.frontier=total_length     DST.frontier=total_vowels
  shared=['shortest_word_length', 'count_all_lowercase']  SRC.frontier=count_short      DST.frontier=total_length
  shared=['all_unique_lengths', 'count_starts_vowel']  SRC.frontier=longest_word_length DST.frontier=shortest_word_length
  shared=['total_uppercase_letters', 'count_all_lowercase']  SRC.frontier=all_unique_lengths DST.frontier=count_short
  shared=['total_length', 'any_empty']  SRC.frontier=count_short      DST.frontier=has_palindrome
  share

╭─ SRC · T3 CACHED-FROM (64, 101) (orange) · pivot@251 · align=pad ───────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the total number of uppercase letters across all words\n                                                    │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) whether all words have distinct lengths\n                                                                   │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     auto a = 0;\n                                                                                               │
│     for(auto &a_w : words) for(char a_c : a_w) if(std::isupper((unsigned char)a_c)) a++;\n                      │
│     auto b = 0;\n                                                                                               │
│     for(auto &b_w : words){ bool b_has = false, b_ok = true; for(char b_c : b_w){ if(std::isalpha((unsigned     │
│ char)b_c)){ b_has = true; if(std::isupper((unsigned char)b_c)) b_ok = false; } } if(b_has && b_ok) b++; }\n     │
│     auto c =                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · T4 PATCHED-TO (63, 100) (orange) · pivot@152 · align=pad ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the total number of uppercase letters across all words\n                                                    │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) the number of words shorter than three characters\n                                                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     b = sum(1 for w in words if w.islower())\n                                                                  │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
mp_dst_tok.input_ids.shape[1], MP_DST_SPAN, mp_dst_tok.input_ids.shape[1] - MP_DST_SPAN[0]

(153, (63, 100), 90)

## Multi-predicate intervention — k+v transplant of the frontier-predicate span

Same `run_kv_transfer_intervention` as the kv-cluster (align the T3/T4 spans per `ALIGN_KNOB`, cache SRC key+value over `PATCH_HEADS`, GQA-corrected headwise routing), scored with `eval_kv_transfer` (causal + readout; no gaze). `ALPHA` applies.

In [13]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
QUERY_POSITIONS = "all"  # which DST query positions of PATCH_HEADS read the transplanted span:
                         #   "all"  = every query after the span (original; pre-pivot latents change too)
                         #   "last" = only the last query of each forward (pivot, then each generated token)
                         #            -> pre-pivot DST latents are untouched
                         #   "last_n" = the last QUERY_LAST_N prompt tokens (ending at the pivot), then each
                         #            generated token -> latents before them are untouched
QUERY_LAST_N = 4         # only used when QUERY_POSITIONS == "last_n" (1 == "last")
QP = QUERY_LAST_N if QUERY_POSITIONS == "last_n" else QUERY_POSITIONS   # what headwise_kv_patch receives
# PATCH_HEADS = GOAL_HEADS_250                    # heads carrying the transplant (GOAL | UNION | ...)
PATCH_HEADS = [(layer_idx, head_idx) for (layer_idx, head_idx) in GOAL_HEADS_250 if layer_idx>=30 or layer_idx==2]
# PATCH_HEADS = [(layer_idx, head_idx) for layer_idx in range(40, 50) for head_idx in range(mt.n_heads)]
READOUT_HEADS = GOAL_HEADS_250                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 64)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------
print(f"{len(PATCH_HEADS)=}")

len(PATCH_HEADS)=230


In [14]:
from src.diff.functional import run_kv_transfer_intervention
from src.diff.eval import eval_kv_transfer


def report(tag, inst, out):
    ev = eval_kv_transfer(
        inst,
        patched_gen=out["gen_patched"],
        readouts=(out["ro_src"], out["ro_dst"], out["ro_patched"]),
    )
    rich_print(
        f"[bold]{tag}[/bold]  SRC.target={inst.expected_next_desc!r}  DST.target={inst.dst_next_desc!r}"
    )
    rich_print(f"  [red1]clean SRC[/red1]    :\n{escape(out['gen_src'])}")
    rich_print(f"  [sea_green3]clean DST[/sea_green3]    :\n{escape(out['gen_dst'])}")
    rich_print(f"  [salmon1]patched DST[/salmon1]  :\n{escape(out['gen_patched'])}")
    rich_print(
        f"  readout src→dst→pat : {out['ro_src']!r} → {out['ro_dst']!r} → {out['ro_patched']!r}"
    )
    rich_print(f"  [bold]eval[/bold] : {ev.summary()}")
    return ev


# k+v transplant of the SRC frontier (T3) description onto the DST frontier (T4) span
report(
    f"multi-predicate · k+v transplant · {inst_mp.meta['datatype']} · align={ALIGN_KNOB}",
    inst_mp,
    run_kv_transfer_intervention(
        mt=mt,
        inst=inst_mp,
        patch_heads=PATCH_HEADS,
        # patch_heads=GOAL_HEADS_250,
        readout_heads=READOUT_HEADS,
        alpha=ALPHA,
        align=ALIGN_KNOB,
        query_positions=QP,
        # ablate_heads=ABLATE_HEADS,
        # ablation_strat=ABLATION_STRAT,
    ),
)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=251 ' =' => Checking for unique elements using a set.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=152 ' =' => I'm checking if the length of the string is less than 3.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=152 ' =' => I'm checking if the current line is empty.

... (3 outputs left out)

patched DST  :
 sum(1 for w in words if len(w) == 1)
    return a, b, c
</code>

readout src→dst→pat : 'Checking for unique elements using a set.' → "I'm checking if the length of the string is 
less than 3." → "I'm checking if the current line is empty."

eval : causal: no (dst) | readout: src→dst→neither

QTransferEval(causal={'label': 'dst', 'why': 'The computation counts words where len(w) == 1, which is a partial implementation of counting words shorter than three characters (sub-goal B), not checking for distinct lengths (sub-goal A).', 'raw': 'The first computation is `sum(1 for w in words if len(w) == 1)`, which counts words with exactly 1 character (length < 3, but specifically length == 1). This is a partial implementation of counting words shorter than 3 characters, though it only counts words of length 1, not length 1 or 2.\n\nHowever, the task asks me to judge by what computation is actually performed. The computation checks `len(w) == 1`, which counts words of length exactly 1. This is a subset of "words shorter than three characters" (which would be `len(w) < 3`). It\'s a partial/approximate implementation of sub-goal B.\n\nSub-goal A would require checking if all words have distinct lengths, which this does not do.\n\n<verdict>B</verdict>\n<why>The computation counts words

## Residual transfer of the shared code prefix — `p_src: a'b'c -> c` vs `p_dst: a'b'd -> d` (py → py)

SRC lists (a, b, c), DST lists (a, b, d); both prefill `a` and `b` and stop at the frontier's `c =`. Everything from the
user turn's `<|im_end|>` to the ` =` pivot is **token-identical**:
```
<|im_end|>\n<|im_start|>assistant\n<code>\ndef func(words: list[str]):\n    a = ...\n    b = ...\n    c =
```
Over this segment, the SRC residual stream (output of **every** decoder layer, `RP_LAYERS`) replaces DST's. The DST task
list (`d`) is untouched; everything after the user turn now carries SRC's computation. Question: does the patched DST write
SRC's `c` (the goal rides in the prefix latents) or its own `d` (it is re-derived from the task span)?

Caveats:
* With the pivot ` =` included (`RP_INCLUDE_PIVOT=True`) and the last layer patched, DST's next-token logits are SRC's
  **by construction**, so the first generated token is copied. Only the tokens after it are informative. `False` leaves
  the pivot clean: it then reads DST's task span and the SRC-patched k/v of the segment.
* `RP_INCLUDE_SRC_FRONT=True` also transfers the frontier-description span (SRC's `c` text -> DST's `d` text). If the two
  differ in token length, the shorter one is padded with `\n` tokens right before it (`align_kv_spans(mode="pad")`), so
  every token of DST's `d` span is overwritten. All three runs (clean SRC, clean DST, patched) then use the padded prompts.
* `RP_MODE="ctxfree"` writes each DST token's **context-free** latents into the window instead of SRC's (the
  pre-commitment ablation recipe: the token alone after `RP_NPAD` pad tokens, pad = EOS `<|im_end|>`, every layer). This removes
  all context-specific information from the window and measures how damaging that is on its own.
* Generated tokens are not patched; they read the KV cache, whose segment k/v come from the patched residuals.
* The GoalScope readout at ` =` reads the goal heads' write at the pivot under the patch. Under `RP_INCLUDE_PIVOT=True` their
  input is SRC's residual (layer l-1) while their keys still include DST's task span.

In [15]:
#------------------------------------------------------------------------------------------------------
RP_SEED, RP_IDX = 0, 4          # python->python instance: multi_predicate_setups(seed=RP_SEED)[RP_IDX]
RP_LAYERS = "all"               # "all" or an iterable of layer indices whose OUTPUT residual is transferred
RP_INCLUDE_PIVOT = False        # False: transfer UNTIL the pivot ` =` (pivot left clean) | True: also the pivot (then the first generated token is SRC's by construction)
RP_INCLUDE_SRC_FRONT = False    # True: also transfer the frontier description (SRC c-span -> DST d-span), "\n"-padded to equal length
RP_MODE = "src"                 # what overwrites the window's residuals: "src" (SRC's latents = residual transfer) |
                                #   "ctxfree" (each DST token's CONTEXT-FREE latents: <pad> x RP_NPAD + tok — all context removed)
RP_NPAD = 15                    # "ctxfree": pads before the token (pad = EOS <|im_end|>; eval_precommit_ablation.py default: 16)
RP_MAX_NEW = 64
RP_ORACLE = True                # eval_kv_transfer (causal + readout oracle) on the patched run
#------------------------------------------------------------------------------------------------------
import baukit
from contextlib import contextmanager


def layer_outputs(tok, positions, layers):
    """{layer: (len(positions), d)} — each decoder layer's OUTPUT residual at `positions` (CPU float32)."""
    names = {l: mt.layer_name_format.format(l) for l in layers}
    with torch.no_grad(), baukit.TraceDict(mt._model, list(names.values()), retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    out = lambda o: o[0] if isinstance(o, tuple) else o
    return {l: out(tr[n].output)[0, positions].detach().float().cpu() for l, n in names.items()}


@contextmanager
def residual_patch(cache, positions, layers):
    """Overwrite each decoder layer's OUTPUT residual at absolute `positions` with `cache[layer]`.
    Only forwards that contain those positions are touched (the prompt pass); decode steps (1 token)
    are skipped — the effect then rides the KV cache, whose k/v at `positions` were computed from the
    patched residuals."""
    handles = []

    def make(src):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            if h.shape[1] <= max(positions):
                return out
            h = h.clone()
            h[:, positions] = src.to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        return hook

    try:
        for l in layers:
            mod = baukit.get_module(mt._model, mt.layer_name_format.format(l))
            handles.append(mod.register_forward_hook(make(cache[l])))
        yield
    finally:
        for hd in handles:
            hd.remove()


def show_windows(tok, title, task_span=None, windows=(), mark_pivot=True):
    """The prompt as the model sees it: task_span -> yellow, every transferred window -> orange,
    pivot -> salmon (only if `mark_pivot`). Text objects, so no markup escaping issues."""
    ids = tok["input_ids"][0].tolist()
    task = set(range(*task_span)) if task_span else set()
    win = {q for lo, hi in windows for q in range(lo, hi)}
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid]).replace("\n", "\\n\n")
        style = ("bold black on salmon1" if mark_pivot and i == len(ids) - 1
                 else "bold black on dark_orange" if i in win else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{escape(title)}[/bold]", title_align="left", border_style="grey42",
                     padding=(1, 1)))

def ctxfree_latents(tids, layers, n_pad):
    """{layer: (len(tids), d)} — CONTEXT-FREE latents (as `BareLatents` in scripts/evaluation/eval_precommit_ablation.py):
    each token alone after `n_pad` pad tokens (pad = tokenizer pad, i.e. EOS under ModelandTokenizer; all positions
    attended; no chat template), every decoder layer's OUTPUT at the token. One batched forward over the unique ids."""
    pad = mt.tokenizer.pad_token_id if mt.tokenizer.pad_token_id is not None else mt.tokenizer.eos_token_id
    uniq = sorted({int(t) for t in tids})
    dev = next(mt._model.parameters()).device
    ids = torch.tensor([[pad] * n_pad + [t] for t in uniq], device=dev)
    names = {l: mt.layer_name_format.format(l) for l in layers}
    with torch.no_grad(), baukit.TraceDict(mt._model, list(names.values()), retain_output=True) as tr:
        mt._model(input_ids=ids, attention_mask=torch.ones_like(ids))
    out = lambda o: o[0] if isinstance(o, tuple) else o
    row = {t: i for i, t in enumerate(uniq)}
    idx = [row[int(t)] for t in tids]
    return {l: out(tr[n].output)[idx, -1].detach().float().cpu() for l, n in names.items()}


In [16]:
mp_py = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=RP_SEED,
    src_lang="python", dst_lang="python", naming="letters", letter_stub="var_eq"
)
for i, s in enumerate(mp_py):
    print(f"[{i:2d}] shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<22} DST.frontier={s.dst_next}")

inst_rp = mp_py[RP_IDX]
mr = D.materialize(inst_rp)

# frontier-description spans = ONLY the frontier item's text (SRC c / DST d), not the whole task list. With
# RP_INCLUDE_SRC_FRONT the shorter one is "\n"-padded at the token level right before it, so both spans have equal
# length; the padded prompts are then used for ALL three runs.
RP_C_DESC, RP_D_DESC = inst_rp.src_frontier.description, inst_rp.dst_frontier.description
assert mr["src_prompt"].count(RP_C_DESC) == 1 and mr["dst_prompt"].count(RP_D_DESC) == 1, "frontier text not unique"
_fs0 = token_span(mr["src_prompt"], RP_C_DESC)
_fd0 = token_span(mr["dst_prompt"], RP_D_DESC)
if RP_INCLUDE_SRC_FRONT:
    rp_src_tok, rp_dst_tok, _, _, RP_SRC_FSPAN, RP_DST_FSPAN = align_kv_spans(
        mt, mr["src_prompt"], mr["dst_prompt"], _fs0, _fd0, mode="pad")
else:
    rp_src_tok = prepare_input(prompts=mr["src_prompt"], tokenizer=mt)
    rp_dst_tok = prepare_input(prompts=mr["dst_prompt"], tokenizer=mt)
    RP_SRC_FSPAN, RP_DST_FSPAN = _fs0, _fd0
src_ids, dst_ids = rp_src_tok["input_ids"][0].tolist(), rp_dst_tok["input_ids"][0].tolist()
L_src, L_dst = len(src_ids), len(dst_ids)

# shared segment = last <|im_end|> (end of the user turn) .. pivot, aligned from the END of each prompt
im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
seg_start = max(i for i, t in enumerate(dst_ids) if t == im_end)
n_seg = L_dst - seg_start
assert src_ids[L_src - n_seg:] == dst_ids[seg_start:], "SRC/DST code prefix is not token-identical"
n_pat = n_seg if RP_INCLUDE_PIVOT else n_seg - 1
seg_src = (L_src - n_seg, L_src - n_seg + n_pat)
seg_dst = (seg_start, seg_start + n_pat)

# transferred windows: [frontier span (pad-aligned)] + segment; SRC/DST positions pair up 1:1 in order
RP_SRC_WIN = ([RP_SRC_FSPAN] if RP_INCLUDE_SRC_FRONT else []) + [seg_src]
RP_DST_WIN = ([RP_DST_FSPAN] if RP_INCLUDE_SRC_FRONT else []) + [seg_dst]
RP_SRC_POS = [q for lo, hi in RP_SRC_WIN for q in range(lo, hi)]
RP_DST_POS = [q for lo, hi in RP_DST_WIN for q in range(lo, hi)]
assert len(RP_SRC_POS) == len(RP_DST_POS), (len(RP_SRC_POS), len(RP_DST_POS))
RP_LAYER_LIST = list(range(mt.n_layer)) if RP_LAYERS == "all" else sorted(RP_LAYERS)
RP_CFG = (RP_SEED, RP_IDX, RP_INCLUDE_PIVOT, RP_INCLUDE_SRC_FRONT)   # the run cell checks the knobs still match

_pad = (_fd0[1] - _fd0[0]) - (_fs0[1] - _fs0[0])
print(f"\nSRC frontier (c) = {RP_C_DESC!r}\nDST frontier (d) = {RP_D_DESC!r}")
print(f"frontier spans: SRC {_fs0[1] - _fs0[0]} tok, DST {_fd0[1] - _fd0[0]} tok"
      + (f" -> {abs(_pad)} '\\n' pad token(s) before the {'SRC' if _pad > 0 else 'DST'} span" if RP_INCLUDE_SRC_FRONT and _pad
         else "" if RP_INCLUDE_SRC_FRONT else " (not transferred)"))
print(f"segment: {n_seg} tokens · transferred {len(RP_DST_POS)} positions × {len(RP_LAYER_LIST)} layers · "
      f"windows SRC {RP_SRC_WIN} -> DST {RP_DST_WIN} · pivot {mt.tokenizer.decode([dst_ids[-1]])!r} "
      f"{'included' if RP_INCLUDE_PIVOT else 'left clean'}")
show_windows(rp_dst_tok, f"DST · task d (yellow) · transferred (orange) · L={L_dst}",
             task_span=RP_DST_FSPAN, windows=RP_DST_WIN, mark_pivot=RP_INCLUDE_PIVOT)

[ 0] shared=['all_unique_lengths', 'has_palindrome']  SRC.frontier=distinct_count         DST.frontier=count_short
[ 1] shared=['any_empty', 'count_ends_consonant']  SRC.frontier=total_uppercase_letters DST.frontier=distinct_count
[ 2] shared=['total_vowels', 'longest_word_length']  SRC.frontier=count_capitalized      DST.frontier=count_short
[ 3] shared=['longest_word_length', 'count_long']  SRC.frontier=any_empty              DST.frontier=total_length
[ 4] shared=['distinct_count', 'total_uppercase_letters']  SRC.frontier=total_length           DST.frontier=total_vowels
[ 5] shared=['shortest_word_length', 'count_all_lowercase']  SRC.frontier=count_short            DST.frontier=total_length
[ 6] shared=['all_unique_lengths', 'count_starts_vowel']  SRC.frontier=longest_word_length    DST.frontier=shortest_word_length
[ 7] shared=['total_uppercase_letters', 'count_all_lowercase']  SRC.frontier=all_unique_lengths     DST.frontier=count_short
[ 8] shared=['total_length', 'any_empty']  SR

╭─ DST · task d (yellow) · transferred (orange) · L=141 ──────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
# ── residual transfer: SRC segment residuals (every layer in RP_LAYER_LIST) -> DST ──
if RP_MODE == "src":        # SRC's latents over the window (residual transfer)
    rp_cache = layer_outputs(rp_src_tok, RP_SRC_POS, RP_LAYER_LIST)
elif RP_MODE == "ctxfree":  # each DST token's context-free latents over the window (context ablation)
    rp_cache = ctxfree_latents([dst_ids[q] for q in RP_DST_POS], RP_LAYER_LIST, RP_NPAD)
else:
    raise ValueError(f"RP_MODE must be 'src' | 'ctxfree', got {RP_MODE!r}")
_pad_id = mt.tokenizer.pad_token_id if mt.tokenizer.pad_token_id is not None else mt.tokenizer.eos_token_id
rich_print(f"RP_MODE={RP_MODE!r} · window {RP_DST_WIN} ({len(RP_DST_POS)} pos) × {len(RP_LAYER_LIST)} layers"
           + (f" · context-free: {mt.tokenizer.decode([_pad_id])!r} × {RP_NPAD} + tok" if RP_MODE == "ctxfree" else ""))
piv_s, piv_d = L_src - 1, L_dst - 1
assert RP_CFG == (RP_SEED, RP_IDX, RP_INCLUDE_PIVOT, RP_INCLUDE_SRC_FRONT), "knobs changed since the setup cell -- re-run it"

# both prompts: frontier task (yellow) · transferred windows (orange) · pivot (salmon, only if transferred)
show_windows(rp_src_tok, f"SRC · frontier c · CACHED-FROM {RP_SRC_WIN} · L={L_src}",
             task_span=RP_SRC_FSPAN, windows=RP_SRC_WIN, mark_pivot=RP_INCLUDE_PIVOT)
show_windows(rp_dst_tok, f"DST · frontier d · PATCHED-TO {RP_DST_WIN} · L={L_dst}",
             task_span=RP_DST_FSPAN, windows=RP_DST_WIN, mark_pivot=RP_INCLUDE_PIVOT)

# sanity: the pivot's latents under the transfer vs the clean DST run — every decoder layer's output residual
_all_layers = list(range(mt.n_layer))
_h_dst = layer_outputs(rp_dst_tok, [piv_d], _all_layers)
with residual_patch(rp_cache, RP_DST_POS, RP_LAYER_LIST):
    _h_pat = layer_outputs(rp_dst_tok, [piv_d], _all_layers)
_l2 = [float((_h_pat[l][0] - _h_dst[l][0]).norm()) for l in _all_layers]
_rel = [_l2[l] / float(_h_dst[l][0].norm()) for l in _all_layers]
rich_print(f"pivot ' =' (pos {piv_d}) · ‖h_patched − h_dst‖₂ per layer output (relative to ‖h_dst‖) · "
           f"pivot transferred: {RP_INCLUDE_PIVOT}")
for i in range(0, len(_all_layers), 8):
    print("  " + "  ".join(f"L{l:02d} {_l2[l]:8.2f} ({_rel[l]:.2f})" for l in _all_layers[i:i + 8]))

rp_gen_src = greedy(rp_src_tok, max_new=RP_MAX_NEW)
rp_gen_dst = greedy(rp_dst_tok, max_new=RP_MAX_NEW)
with residual_patch(rp_cache, RP_DST_POS, RP_LAYER_LIST):
    rp_gen_patched = greedy(rp_dst_tok, max_new=RP_MAX_NEW)


def rp_iface(mt_, inputs, patches=None, model_kwargs=None):
    with residual_patch(rp_cache, RP_DST_POS, RP_LAYER_LIST):
        return mt_._model(**dict(inputs), **dict(model_kwargs or {}))


rp_ro_src = goalscope_readout("   ↳ goalscope · source_run", rp_src_tok, piv_s, heads=READOUT_HEADS)[0]["patched_gen"]
rp_ro_dst = goalscope_readout("   ↳ goalscope · destination_run", rp_dst_tok, piv_d, heads=READOUT_HEADS)[0]["patched_gen"]
rp_ro_patched = goalscope_readout("   ↳ goalscope · patched_run", rp_dst_tok, piv_d, heads=READOUT_HEADS,
                                  iface=rp_iface)[0]["patched_gen"]

for tag, col, gen, ro in (("clean SRC (c)", "red1", rp_gen_src, rp_ro_src),
                          ("clean DST (d)", "sea_green3", rp_gen_dst, rp_ro_dst),
                          (f"patched DST · {RP_MODE}", "salmon1", rp_gen_patched, rp_ro_patched)):
    rich_print(Panel(Text(gen), title=f"[{col}]{tag}[/{col}] · goalscope @ ' =': [italic]{escape(ro)}[/italic]",
                     title_align="left", border_style="grey42"))
print(f"SRC frontier (c) = {mr['src_task_text']!r}\nDST frontier (d) = {mr['dst_task_text']!r}")

if RP_ORACLE:
    from src.diff.eval import eval_kv_transfer
    _rev = eval_kv_transfer(inst_rp, patched_gen=rp_gen_patched, readouts=(rp_ro_src, rp_ro_dst, rp_ro_patched))
    rich_print(f"[bold]eval · residual transfer[/bold] : {_rev.summary()}")

RP_MODE='src' · window [(97, 140)] (43 pos) × 64 layers

╭─ SRC · frontier c · CACHED-FROM [(97, 140)] · L=141 ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of characters across all words\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · frontier d · PATCHED-TO [(97, 140)] · L=141 ─────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

pivot ' =' (pos 140) · ‖h_patched − h_dst‖₂ per layer output (relative to ‖h_dst‖) · pivot transferred: False

  L00     0.00 (0.00)  L01     0.14 (0.00)  L02     0.24 (0.00)  L03     0.35 (0.01)  L04     0.55 (0.01)  L05     0.66 (0.01)  L06     0.70 (0.01)  L07     0.73 (0.01)
  L08     0.76 (0.01)  L09     0.76 (0.01)  L10     0.82 (0.01)  L11     1.16 (0.01)  L12     1.24 (0.01)  L13     1.29 (0.01)  L14     1.33 (0.01)  L15     1.84 (0.01)
  L16     2.16 (0.01)  L17     2.35 (0.01)  L18     2.61 (0.01)  L19     2.62 (0.01)  L20     3.25 (0.02)  L21     3.28 (0.02)  L22     3.38 (0.02)  L23     3.40 (0.02)
  L24     3.46 (0.02)  L25     4.26 (0.02)  L26     4.78 (0.02)  L27     5.22 (0.03)  L28     5.92 (0.03)  L29     7.13 (0.03)  L30     9.82 (0.04)  L31    10.86 (0.04)
  L32    11.23 (0.04)  L33    11.09 (0.04)  L34    12.04 (0.05)  L35    12.28 (0.05)  L36    12.71 (0.05)  L37    27.42 (0.11)  L38    27.82 (0.11)  L39    28.71 (0.11)
  L40    31.06 (0.12)  L41    32.59 (0.11)  L42    39.86 (0.13)  L43    52.15 (0.17)  L44    57.30 (0.19)  L45    63.12 (0.19)  L46    69.18 (0.20)  L47   

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

╭─ clean DST (d) · goalscope @ ' =': I'm counting vowels in the string. ──────────────────────────────────────────╮
│  sum(1 for w in words for c in w if c.lower() in 'aeiou')                                                       │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST · src · goalscope @ ' =': I'm counting vowels in the string. ──────────────────────────────────────╮
│  sum(1 for w in words for c in w if c.lower() in 'aeiou')                                                       │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC frontier (c) = '(1) the number of distinct words\n(2) the total number of uppercase letters across all words\n(3) the total number of characters across all words'
DST frontier (d) = '(1) the number of distinct words\n(2) the total number of uppercase letters across all words\n(3) the total number of vowels across all words'


eval · residual transfer : causal: no (dst) | readout: src→dst→dst

## Path patching — task-span k/v (I1) · progress restore (I2) · goal-head k/v from the pivot (I3) (py → py)

Same `a'b'c -> c` (SRC) vs `a'b'd -> d` (DST) instances as the residual-transfer section. One source k/v transplant over the
task span (`PP_SPAN`, `\n`-padded to equal length), read by two head routes, plus a residual restore. Each step has an
on/off knob:

* **I1 · `TASK_PATCH_HEADS`** read SRC's task-span key/value. Query rule `PP_I1_QUERY="all"` (as `QUERY_POSITIONS="all"`): every
  query after the span start, including the queries **inside** the task span, so DST's task-span latents change. Note that
  `"all"` also covers the pivot and every generated token, so these heads read SRC directly there as well.
* **I2 · progress restore.** I1 changes every later token. From `<|im_end|>` (`PP_I2_FROM="im_end"`) to the frontier variable
  `{var}` (pivot ` =` excluded), every decoder layer's output residual is reset to the **clean DST run**, so these positions'
  k/v are clean too. This **blocks** I1's path through the code progress. `"span_end"` also restores the few tokens between the
  task span and `<|im_end|>` (otherwise I1 can still reach the pivot through them).
* **I2 donor (`PP_I2_DONOR`).** `"res"` (default) restores from a third prompt **p_res: a'b'e -> e**, with the same shared a, b and
  language, and a frontier e not in {c, d} (`PP_RES_FRONTIER`, else a seeded compatible pick). Its `<|im_end|> … {var}` prefill is
  token-identical to DST's (asserted). The window then carries neither d (DST) nor c (SRC). `"dst"` restores from the clean DST run.
* **I3 · `GOAL_PATCH_HEADS`** read SRC's task-span k/v **from the pivot on** (`PP_I3_QUERY="last"`: the pivot on the prompt pass,
  then each generated token). `PP_ALPHA` optionally boosts their OV write at `-1` (1.0 = off). `PP_I3_V_SCALE` multiplies
  the transplanted SRC task-span **values** they read (keys, and so their attention pattern, unchanged; 1.0 = off). With
  2.0 this is a cheap alternative to I2: it boosts the source-span share of their output instead of cleaning the progress.

I1 and I3 share one transplant (`headwise_kv_patch_routes`); a head in both reads the source at the union of the two rules.
I2 acts on the prompt pass only; generated tokens read its effect through the KV cache. All three runs (clean SRC, clean DST,
patched) use the padded prompts. The GoalScope readout at ` =` runs under the same interventions.

In [71]:
len(GOAL_HEADS)

122

### Failed cases from the batch run — pick one for `PP_MANUAL`

Reads `scripts/diff/path_patch_eval.py` results (`results/diff/<model>/path_patch/<run_tag>/<dt>__<src>-<dst>/results.json`)
for one config and lists the instances whose **causal shift missed**, each with a ready-to-paste `PP_MANUAL` dict. No model needed.

In [120]:
FF_CONFIG = "I1-off__I2-off__I3-on__win-span__"     # run-tag prefix of the config to inspect (smoke runs are skipped)
FF_DATATYPE = "list[str]"                          # list[str] | list[int] | int | str
FF_DIRECTION = "python-cpp"                     # python-python | python-cpp | cpp-python | cpp-cpp
FF_SHOW = 10

import glob
_root = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "diff", P.MODEL_NAME.split("/")[-1], "path_patch")
_runs = [d for d in glob.glob(os.path.join(_root, FF_CONFIG + "*")) if "smoke" not in d]
assert len(_runs) == 1, f"FF_CONFIG matches {len(_runs)} run dirs: {[os.path.basename(d) for d in _runs]}"
_fp = os.path.join(_runs[0], f"{FF_DATATYPE.replace('[', '_').replace(']', '')}__{FF_DIRECTION}", "results.json")
_recs = json.load(open(_fp))
_fails = [r for r in _recs if r.get("causal") and not r["causal"]["triggered"]]
print(f"{os.path.basename(_runs[0])}\n{FF_DATATYPE} · {FF_DIRECTION}: {len(_fails)}/{len(_recs)} failed (causal miss)\n")
FF_MANUAL = []   # ready PP_MANUAL dicts, aligned with _fails -> use PP_MANUAL = FF_MANUAL[i] (or PP_MANUAL = _fails[i])
for r in _fails:
    _a, _b = r["group"].split(":", 1)[1].split("-")
    FF_MANUAL.append(dict(datatype=r["datatype"], a=_a, b=_b, c=r["expected_next"], d=r["dst_next"],
                          src_lang=r["src_lang"], dst_lang=r["dst_lang"]))
for i, (r, m) in enumerate(zip(_fails[:FF_SHOW], FF_MANUAL)):
    _first = next((ln for ln in r["raw"]["gen_patched"].splitlines() if ln.strip()), "").strip()
    print(f"[{i}] c={r['expected_next']} -> d={r['dst_next']} · judged {r['causal']['label']} · patched: {_first[:80]!r}")
    print(f"    PP_MANUAL = {m!r}")

I1-off__I2-off__I3-on__win-span__span-list__a1__th-goal_250__gh-goal_250__ro-goal_250
list[str] · python-cpp: 19/32 failed (causal miss)

[0] c=any_empty -> d=count_starts_vowel · judged dst · patched: '0;'
    PP_MANUAL = {'datatype': 'list[str]', 'a': 'distinct_count', 'b': 'total_length', 'c': 'any_empty', 'd': 'count_starts_vowel', 'src_lang': 'python', 'dst_lang': 'cpp'}
[1] c=has_palindrome -> d=total_length · judged dst · patched: '0;'
    PP_MANUAL = {'datatype': 'list[str]', 'a': 'count_all_lowercase', 'b': 'count_short', 'c': 'has_palindrome', 'd': 'total_length', 'src_lang': 'python', 'dst_lang': 'cpp'}
[2] c=total_length -> d=total_vowels · judged dst · patched: '0;'
    PP_MANUAL = {'datatype': 'list[str]', 'a': 'all_unique_lengths', 'b': 'distinct_count', 'c': 'total_length', 'd': 'total_vowels', 'src_lang': 'python', 'dst_lang': 'cpp'}
[3] c=longest_word_length -> d=total_length · judged dst · patched: '0;'
    PP_MANUAL = {'datatype': 'list[str]', 'a': 'shortest_word_le

In [129]:
#------------------------------------------------------------------------------------------------------
PP_SEED, PP_IDX = 1123, 11          # python->python instance: multi_predicate_setups(seed=PP_SEED)[PP_IDX]
PP_MANUAL = _fails[5]                # hand-picked tasks instead of the sampled instance (None = use PP_SEED / PP_IDX), e.g.
#   PP_MANUAL = dict(datatype="list[str]", a="count_long", b="has_palindrome", c="distinct_count", d="count_short",
#                    src_lang="python", dst_lang="python")   # a, b shared (prefilled); c = SRC frontier; d = DST frontier
#   (ids: D.QTASKS[datatype].predicates; the failure-finder cell below prints ready-made dicts)
PP_SPAN = "list"                # task span transplanted: "list" (whole task list = kv-transfer convention) | "frontier" (frontier item)
# I1 — task-span k/v
PP_I1 = False
# TASK_PATCH_HEADS = GOAL_HEADS_250
TASK_PATCH_HEADS = [(layer_idx, head_idx) for (layer_idx, head_idx) in GOAL_HEADS_250 if layer_idx == 0]
PP_I1_QUERY = "all"             # "all" (spec: queries inside the span and after) | "last" | int n — see headwise_kv_patch
# I2 — restore clean-DST residuals over the code progress (pivot excluded)
PP_I2 = False
PP_I2_FROM = "im_end"           # "im_end" (from <|im_end|>) | "span_end" (from right after the task span)
PP_I2_LAYERS = "all"            # "all" or an iterable of layers whose OUTPUT residual is restored
PP_I2_DONOR = "res"             # I2 residual donor: "res" = p_res a'b'e -> e (frontier e not in {c, d}) | "dst" = clean DST
PP_RES_FRONTIER = None          # e's predicate id (e.g. "count_long"); None = seeded pick among compatible predicates
# I3 — goal heads read the source k/v from the pivot on
PP_I3 = True
# GOAL_PATCH_HEADS = GOAL_HEADS_250
# GOAL_PATCH_HEADS = GOAL_HEADS
GOAL_PATCH_HEADS = [(layer_idx, head_idx) for (layer_idx, head_idx) in GOAL_HEADS_250 if layer_idx>=30]
PP_I3_QUERY = "last"            # "last" = pivot, then each generated token
PP_I3_V_SCALE = 4.0             # x the transplanted SRC task-span VALUES read by GOAL_PATCH_HEADS (keys / attention unchanged);
                                #   2.0 = the cheap alternative to I2 for "I2 off · task span k/v"; 1.0 = off
PP_ALPHA = 1.0                  # OV boost of GOAL_PATCH_HEADS at -1 (pivot / each generated token) when PP_I3; 1.0 = off
PP_MAX_NEW = 64
PP_ORACLE = True                # eval_kv_transfer (causal + readout oracle) on the patched run

READOUT_HEADS = GOAL_PATCH_HEADS                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
#------------------------------------------------------------------------------------------------------
from contextlib import ExitStack
from src.diff.functional import headwise_kv_patch_routes, heads_per_layer, cache_src_proj, boost_head_ov
assert "residual_patch" in globals() and "layer_outputs" in globals() and "show_windows" in globals(), \
    "run the residual-transfer knob cell first (defines layer_outputs / residual_patch / show_windows)"

In [130]:
# PP_MANUAL

In [131]:
pp_mp = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=PP_SEED,
    src_lang="python", dst_lang="cpp", naming="letters", letter_stub="var_eq"
)
inst_pp = pp_mp[PP_IDX]


def make_manual_instance(template, a, b, c, d, datatype=None, src_lang=None, dst_lang=None):
    """The sampled instance's format (letters naming, var_eq pivot, listed presentation) with hand-picked predicates:
    a, b shared (prefilled), c = SRC frontier, d = DST frontier. datatype / languages default to the template's."""
    import dataclasses
    dt = datatype or template.datatype
    bank = D.QTASKS[dt]
    by_id = {p.id: p for p in bank.predicates}
    ids = [a, b, c, d]
    missing = [x for x in ids if x not in by_id]
    assert not missing, f"unknown predicate id(s) {missing} for {dt}; choose from {sorted(by_id)}"
    assert len(set(ids)) == 4, "a, b, c, d must be distinct"
    bad = [(x, y) for i, x in enumerate(ids) for y in ids[i + 1:]
           if not (D.qpred_compatible(dt, x, y) and D.qpred_compatible(dt, y, x))]
    if bad:
        print(f"warning: incompatible pair(s) {bad} (the sampler would never draw these together)")
    return dataclasses.replace(template, datatype=dt, shared=(by_id[a], by_id[b]), src_frontier=by_id[c],
                               dst_frontier=by_id[d], var_name=bank.var_names[0],
                               src_lang=src_lang or template.src_lang, dst_lang=dst_lang or template.dst_lang,
                               meta={**template.meta, "datatype": dt, "dst_next": d, "manual": True})


def manual_from_record(r):
    """A path_patch_eval result record (e.g. the finder's `_fails[i]`) -> the PP_MANUAL dict of its instance."""
    a, b = r["group"].split(":", 1)[1].split("-")
    return dict(datatype=r["datatype"], a=a, b=b, c=r["expected_next"], d=r["dst_next"],
                src_lang=r["src_lang"], dst_lang=r["dst_lang"])


if PP_MANUAL:   # a PP_MANUAL dict, or a raw batch record (`_fails[i]`)
    _man = manual_from_record(PP_MANUAL) if "expected_next" in PP_MANUAL else PP_MANUAL
    inst_pp = make_manual_instance(inst_pp, **_man)
_pp_manual_used = repr(PP_MANUAL)
print(f"instance{' (MANUAL)' if PP_MANUAL else f' [{PP_IDX}]'}: {inst_pp.datatype} · {inst_pp.src_lang} -> {inst_pp.dst_lang} · "
      f"a={inst_pp.shared[0].id} b={inst_pp.shared[1].id} · c(SRC)={inst_pp.src_frontier.id} · d(DST)={inst_pp.dst_frontier.id}")
mp_ = D.materialize(inst_pp)
PP_C_DESC, PP_D_DESC = inst_pp.src_frontier.description, inst_pp.dst_frontier.description
_txt_s, _txt_d = ((mp_["src_task_text"], mp_["dst_task_text"]) if PP_SPAN == "list" else (PP_C_DESC, PP_D_DESC))
assert mp_["src_prompt"].count(_txt_s) == 1 and mp_["dst_prompt"].count(_txt_d) == 1, "task span text not unique"
_s0, _d0 = token_span(mp_["src_prompt"], _txt_s), token_span(mp_["dst_prompt"], _txt_d)
pp_src_tok, pp_dst_tok, PP_SRC_POS, PP_DST_POS, PP_SRC_SPAN, PP_DST_SPAN = align_kv_spans(
    mt, mp_["src_prompt"], mp_["dst_prompt"], _s0, _d0, mode="pad")
pp_dst_ids = pp_dst_tok["input_ids"][0].tolist()
PP_L_SRC, PP_L_DST = pp_src_tok["input_ids"].shape[1], len(pp_dst_ids)

# I2 window: <|im_end|> (or the end of the task span) .. {var}; the pivot ` =` (last token) is excluded
_im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
_i2_lo = (max(i for i, t in enumerate(pp_dst_ids) if t == _im_end) if PP_I2_FROM == "im_end" else PP_DST_SPAN[1])
PP_SEG = list(range(_i2_lo, PP_L_DST - 1))
PP_I2_LAYER_LIST = list(range(mt.n_layer)) if PP_I2_LAYERS == "all" else sorted(PP_I2_LAYERS)
PP_CFG = (PP_SEED, PP_IDX, PP_SPAN, PP_I2_FROM)   # the run cell checks these still match

print(f"SRC frontier (c) = {PP_C_DESC!r}\nDST frontier (d) = {PP_D_DESC!r}")
print(f"I1 task span ({PP_SPAN}): SRC {tuple(PP_SRC_SPAN)} -> DST {tuple(PP_DST_SPAN)} ({len(PP_DST_POS)} tok, pad-aligned) · "
      f"I2 restore: DST {PP_SEG[0]}..{PP_SEG[-1]} ({len(PP_SEG)} tok, "
      f"{mt.tokenizer.decode([pp_dst_ids[PP_SEG[0]]])!r} .. {mt.tokenizer.decode([pp_dst_ids[PP_SEG[-1]]])!r}) · "
      f"pivot {PP_L_DST - 1} {mt.tokenizer.decode([pp_dst_ids[-1]])!r}")
_gap = PP_SEG[0] - PP_DST_SPAN[1]
if _gap > 0:
    print(f"  not restored between the task span and I2: {_gap} tok {mt.tokenizer.decode(pp_dst_ids[PP_DST_SPAN[1]:PP_SEG[0]])!r}")
_ov = set(map(tuple, TASK_PATCH_HEADS)) & set(map(tuple, GOAL_PATCH_HEADS))
print(f"heads: TASK_PATCH_HEADS={len(TASK_PATCH_HEADS)} · GOAL_PATCH_HEADS={len(GOAL_PATCH_HEADS)} · in both: {len(_ov)}")
show_windows(pp_src_tok, f"SRC · task span k/v CACHED-FROM {tuple(PP_SRC_SPAN)} (orange) · pivot (salmon) · L={PP_L_SRC}",
             windows=[tuple(PP_SRC_SPAN)], mark_pivot=True)
show_windows(pp_dst_tok, f"DST · I1 task span PATCHED-TO {tuple(PP_DST_SPAN)} (orange) · I2 restored progress "
             f"{PP_SEG[0]}..{PP_SEG[-1]} (yellow) · pivot (salmon) · L={PP_L_DST}",
             task_span=(PP_SEG[0], PP_SEG[-1] + 1), windows=[tuple(PP_DST_SPAN)], mark_pivot=True)

# ── I2 residual donor p_res: a'b'e -> e — same shared a, b and language; frontier e not in {c, d} ──
import dataclasses, random
_dt = inst_pp.datatype
_used = [p.id for p in inst_pp.shared] + [inst_pp.src_frontier.id, inst_pp.dst_frontier.id]
_cands = [p for p in D.QTASKS[_dt].predicates
          if p.id not in _used and all(D.qpred_compatible(_dt, p.id, u) and D.qpred_compatible(_dt, u, p.id) for u in _used)]
if PP_RES_FRONTIER is None:
    _e = random.Random(1000 * PP_SEED + PP_IDX).choice(_cands)
else:
    _e = next((p for p in _cands if p.id == PP_RES_FRONTIER), None)
    assert _e is not None, f"{PP_RES_FRONTIER!r} is not an allowed e; choose from {[p.id for p in _cands]}"
# p_res is rendered from an instance's SRC side, so give that side DST's language: res_lang = dst_lang (p_src's
# language is irrelevant here — it only supplies the k/v for I1 / I3)
inst_res = dataclasses.replace(inst_pp, src_frontier=_e, src_lang=inst_pp.dst_lang)   # its SRC side = a, b, e
assert inst_res.src_lang == inst_pp.dst_lang, "p_res must be in DST's language to share its progress prefill"
mres = D.materialize(inst_res)
pp_res_tok = prepare_input(prompts=mres["src_prompt"], tokenizer=mt)
pp_res_ids = pp_res_tok["input_ids"][0].tolist()
PP_L_RES = len(pp_res_ids)
PP_RES_SEG = [PP_L_RES - PP_L_DST + q for q in PP_SEG]                   # DST's I2 window, aligned from the END
assert [pp_res_ids[q] for q in PP_RES_SEG] == [pp_dst_ids[q] for q in PP_SEG], "p_res / p_dst progress tokens differ"
assert pp_res_ids[-1] == pp_dst_ids[-1], "p_res pivot differs"
PP_E_DESC, _pp_res_knob = _e.description, PP_RES_FRONTIER


def pp_i2_cache():
    """The I2 restore cache over PP_SEG (rows in PP_SEG order) from the donor PP_I2_DONOR."""
    assert _pp_res_knob == PP_RES_FRONTIER, "PP_RES_FRONTIER changed since the setup cell -- re-run it"
    if PP_I2_DONOR == "res":
        return layer_outputs(pp_res_tok, PP_RES_SEG, PP_I2_LAYER_LIST)
    if PP_I2_DONOR == "dst":
        return layer_outputs(pp_dst_tok, PP_SEG, PP_I2_LAYER_LIST)
    raise ValueError(f"PP_I2_DONOR must be 'res' | 'dst', got {PP_I2_DONOR!r}")


print(f"\np_res frontier (e) = {PP_E_DESC!r} ({_e.id}; {len(_cands)} allowed) · I2 donor = {PP_I2_DONOR!r} · "
      f"p_res window {PP_RES_SEG[0]}..{PP_RES_SEG[-1]} == DST {PP_SEG[0]}..{PP_SEG[-1]} (token-identical)")
show_windows(pp_res_tok, f"p_res · frontier e · I2 donor window {PP_RES_SEG[0]}..{PP_RES_SEG[-1]} (yellow) · L={PP_L_RES}",
             task_span=(PP_RES_SEG[0], PP_RES_SEG[-1] + 1), mark_pivot=False)


instance (MANUAL): list[str] · python -> cpp · a=all_unique_lengths b=count_capitalized · c(SRC)=count_short · d(DST)=shortest_word_length
SRC frontier (c) = 'the number of words shorter than three characters'
DST frontier (d) = 'the length of the shortest word'
I1 task span (list): SRC (63, 93) -> DST (64, 94) (30 tok, pad-aligned) · I2 restore: DST 106..202 (97 tok, '<|im_end|>' .. ' c') · pivot 203 ' ='
  not restored between the task span and I2: 12 tok '\nAssume that all the libraries have been already imported.\n'
heads: TASK_PATCH_HEADS=12 · GOAL_PATCH_HEADS=230 · in both: 0


╭─ SRC · task span k/v CACHED-FROM (63, 93) (orange) · pivot (salmon) · L=147 ────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) whether all words have distinct lengths\n                                                                   │
│ (2) the number of capitalized words\n                                                                           │
│ (3) the number of words shorter than three characters\n                                                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(len(w) for w in words)) == len(words)\n                                                         │
│     b = sum(1 for w in words if w[:1].isupper())\n                                                              │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · I1 task span PATCHED-TO (64, 94) (orange) · I2 restored progress 106..202 (yellow) · pivot (salmon) · L=─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) whether all words have distinct lengths\n                                                                   │
│ (2) the number of capitalized words\n                                                                           │
│ (3) the length of the shortest word\n                                                                           │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::set<int> a_s;\n                                                                                        │
│     for(auto &a_w : words) a_s.insert((int)a_w.size());\n                                                       │
│     auto a = a_s.size() == words.size();\n                                                                      │
│     auto b = 0;\n                                                                                               │
│     for (auto& w : words) if (!w.empty() && std::isupper((unsigned char)w[0])) b++;\n                           │
│     auto c =                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


p_res frontier (e) = 'the number of distinct words' (distinct_count; 11 allowed) · I2 donor = 'res' · p_res window 103..199 == DST 106..202 (token-identical)


╭─ p_res · frontier e · I2 donor window 103..199 (yellow) · L=201 ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ (1) whether all words have distinct lengths\n                                                                   │
│ (2) the number of capitalized words\n                                                                           │
│ (3) the number of distinct words\n                                                                              │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::set<int> a_s;\n                                                                                        │
│     for(auto &a_w : words) a_s.insert((int)a_w.size());\n                                                       │
│     auto a = a_s.size() == words.size();\n                                                                      │
│     auto b = 0;\n                                                                                               │
│     for (auto& w : words) if (!w.empty() && std::isupper((unsigned char)w[0])) b++;\n                           │
│     auto c =                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [132]:
# ── path patching: I1 (task-span k/v) + I2 (restore clean-DST progress) + I3 (goal heads read SRC from the pivot) ──
assert PP_CFG == (PP_SEED, PP_IDX, PP_SPAN, PP_I2_FROM), "knobs changed since the setup cell -- re-run it"
assert repr(PP_MANUAL) == _pp_manual_used, "PP_MANUAL changed since the setup cell -- re-run it"
_routes = ([(TASK_PATCH_HEADS, PP_I1_QUERY)] if PP_I1 else []) + ([(GOAL_PATCH_HEADS, PP_I3_QUERY, {"v_scale": PP_I3_V_SCALE})] if PP_I3 else [])
if _routes:   # one SRC k/v cache over the kv-heads of every enabled route
    _kv_heads = [h for r in _routes for h in r[0]]   # routes are (heads, query[, opts])
    _kv_layers = sorted(heads_per_layer(_kv_heads, "k", layers={l for l, _ in _kv_heads}))
    pp_k = cache_src_proj(pp_src_tok, PP_SRC_POS, _kv_layers, "k")
    pp_v = cache_src_proj(pp_src_tok, PP_SRC_POS, _kv_layers, "v")
pp_restore = pp_i2_cache() if PP_I2 else None   # I2 donor: p_res (a'b'e) or clean DST, per PP_I2_DONOR


def pp_stack():
    """The enabled interventions for ONE forward / generation (single-use context managers)."""
    s = ExitStack()
    if _routes:
        s.enter_context(headwise_kv_patch_routes(mt, _routes, pp_k, pp_v, PP_DST_POS))
    if PP_I2:
        s.enter_context(residual_patch(pp_restore, PP_SEG, PP_I2_LAYER_LIST))
    if PP_I3 and PP_ALPHA != 1.0:
        s.enter_context(boost_head_ov(mt, GOAL_PATCH_HEADS, PP_ALPHA, positions=(-1,)))
    return s


def pp_iface(mt_, inputs, patches=None, model_kwargs=None):
    with pp_stack():
        return mt_._model(**dict(inputs), **dict(model_kwargs or {}))


# sanity: residual change vs clean DST (every layer) on the I2 window and at the pivot
_all = list(range(PP_L_DST))
_h0 = layer_outputs(pp_dst_tok, _all, list(range(mt.n_layer)))
with pp_stack():
    _h1 = layer_outputs(pp_dst_tok, _all, list(range(mt.n_layer)))
_d = torch.stack([(_h1[l] - _h0[l]).abs().amax(-1) for l in range(mt.n_layer)])     # (layers, T)
rich_print(f"on: I1={PP_I1} I2={PP_I2} I3={PP_I3} (V×{PP_I3_V_SCALE:g}) · max|Δresid| vs clean DST — task span {_d[:, PP_DST_SPAN[0]:PP_DST_SPAN[1]].max():.3g} · "
           f"I2 window {_d[:, PP_SEG].max():.3g} (donor {PP_I2_DONOR!r}; 0 only for 'dst') · pivot {_d[:, -1].max():.3g}")

pp_gen_src = greedy(pp_src_tok, max_new=PP_MAX_NEW)
pp_gen_dst = greedy(pp_dst_tok, max_new=PP_MAX_NEW)
with pp_stack():
    pp_gen_patched = greedy(pp_dst_tok, max_new=PP_MAX_NEW)

pp_ro_src = goalscope_readout("   ↳ goalscope · source_run", pp_src_tok, PP_L_SRC - 1, heads=READOUT_HEADS)[0]["patched_gen"]
pp_ro_dst = goalscope_readout("   ↳ goalscope · destination_run", pp_dst_tok, PP_L_DST - 1, heads=READOUT_HEADS)[0]["patched_gen"]
pp_ro_patched = goalscope_readout("   ↳ goalscope · patched_run", pp_dst_tok, PP_L_DST - 1, heads=READOUT_HEADS,
                                  iface=pp_iface)[0]["patched_gen"]

for tag, col, gen, ro in (("clean SRC (c)", "red1", pp_gen_src, pp_ro_src),
                          ("clean DST (d)", "sea_green3", pp_gen_dst, pp_ro_dst),
                          (f"patched DST · I1={PP_I1} I2={PP_I2} I3={PP_I3}", "salmon1", pp_gen_patched, pp_ro_patched)):
    rich_print(Panel(Text(gen), title=f"[{col}]{escape(tag)}[/{col}] · goalscope @ ' =': [italic]{escape(ro)}[/italic]",
                     title_align="left", border_style="grey42"))
print(f"SRC frontier (c) = {PP_C_DESC!r}\nDST frontier (d) = {PP_D_DESC!r}")

if PP_ORACLE:
    from src.diff.eval import eval_kv_transfer
    _pev = eval_kv_transfer(inst_pp, patched_gen=pp_gen_patched, readouts=(pp_ro_src, pp_ro_dst, pp_ro_patched))
    rich_print(f"[bold]eval · path patching[/bold] : {_pev.summary()}")

on: I1=False I2=False I3=True (V×4) · max|Δresid| vs clean DST — task span 0 · I2 window 0 (donor 'res'; 0 only for
'dst') · pivot 344

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=146 ' =' => I'm counting lines that have fewer than 3 characters.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=203 ' =' => Identifying the shortest string length in the list.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

╭─ clean DST (d) · goalscope @ ' =': Identifying the shortest string length in the list. ─────────────────────────╮
│  std::min_element(words.begin(), words.end(), [](const std::string& a, const std::string& b) { return a.size()  │
│ < b.size(); })->size();                                                                                         │
│     return std::make_tuple(a, b, c);                                                                            │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST · I1=False I2=False I3=True · goalscope @ ' =': I'm counting the number of elements in the list th─╮
│  std::count_if(words.begin(), words.end(), [](const std::string& s){ return s.size() < 3; });                   │
│     return std::make_tuple(a, b, c);                                                                            │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC frontier (c) = 'the number of words shorter than three characters'
DST frontier (d) = 'the length of the shortest word'


eval · path patching : causal: TRIGGERED (src) | readout: src→dst→src

## Path-patching variants — what the goal heads read at ` =`

Uses the instance, padded prompts and I2 window of the path-patching setup cell, and its switches / head sets
(`PP_I1`, `PP_I2`, `PP_I3`, `TASK_PATCH_HEADS`, `GOAL_PATCH_HEADS`, `PP_ALPHA`, `READOUT_HEADS`). Variants change only
**what is transplanted**, **who can see the current token**, and **where the prompt ends**:

| variant | prompt ends | transplanted SRC k/v (read by I1 / I3 heads) | I3 heads |
|---|---|---|---|
| 1 | with ` =` | task span | normal |
| 2 | with ` =` | task span | **blind to self**: cannot attend to the current token's own key |
| 3 | with ` =` | task span **+ progress**: one contiguous window from the task-span start to ` =` (`PP_V3_PIVOT` includes ` =`; generated tokens are not transplanted). **A1**: the shorter SRC / DST window is `\n`-padded right before it, so this works across languages | normal |
| 5 | with ` =` | task span, **values × 2** for the I3 heads (`v_scale`; the other variants use `PP_I3_V_SCALE`) | normal |
| 4 | **before** ` =` | task span | normal; ` =` is fed as the first decode step, so its latents are built under I1 and I2 (prompt pass) and I3 |

Expectation for 4: it is **numerically the same computation as 1** (up to bf16 differences between one batched prompt pass and
incremental decoding). In 1 the pivot is computed in the same forward as the restored I2 window and reads its restored
k/v layer by layer, which is exactly what decoding ` =` over the I1+I2 cache does. GoalScope extracts in one forward, so
variant 4's readout uses the same forward as variant 1. Any gap between 1 and 4 in the generations is therefore numerical.

In [90]:
#------------------------------------------------------------------------------------------------------
PP_COMPARE = [1, 2, 3, 4, 5]       # which variants to run (each = 1 patched generation + 1 readout [+ oracle])
# variant 3 (A1): ONE window = task span .. progress (contiguous), "\n"-padded to equal SRC / DST length -> any language pair
PP_V3_PIVOT = True              # variant 3: also transplant ` =` itself (the pivot's own key/value is then SRC's)
PP_V4_FORCE_PIVOT = True        # variant 4: feed ` =` as the first decode token (False: the model's own pick at `{var}`)
PP_VARIANTS = {
    1: dict(name="span k/v", window="span", blind_self=False, prefill="upto"),
    2: dict(name="span k/v · I3 blind to self", window="span", blind_self=True, prefill="upto"),
    3: dict(name="span + progress k/v", window="span+progress", blind_self=False, prefill="upto"),
    4: dict(name="prefill until ` =` · ` =` decoded", window="span", blind_self=False, prefill="until"),
    5: dict(name="span k/v · I3 V×2", window="span", blind_self=False, prefill="upto", v_scale=2.0),
}
#------------------------------------------------------------------------------------------------------

In [91]:
# ── variants: one generator with separate prompt-pass / decode-step interventions ──
from rich.table import Table
from rich.console import Console


def pp_geometry(cfg):
    """Token sequences + positions for one variant.

    window="span": the setup cell's task-span alignment (pp_src_tok / pp_dst_tok, PP_SEG, pp_restore_v).
    window="span+progress" (A1): ONE contiguous window = task span .. end of the progress (incl. ` =` if PP_V3_PIVOT),
    re-aligned from the RAW prompts with align_kv_spans(mode="pad"): the shorter window gets "\n" tokens spliced
    right before it, so SRC and DST windows have equal length and T1 / T2 stay position-aligned even when SRC is
    another language. DST may then carry a different padding than in the setup cell, so the I2 window and its restore
    cache are rebuilt in these coordinates."""
    if cfg["window"] == "span":
        return dict(src_tok=pp_src_tok, dst_tok=pp_dst_tok, src_w=list(PP_SRC_POS), dst_w=list(PP_DST_POS),
                    seg=PP_SEG, restore=pp_restore_v, note="")
    s0, d0 = token_span(mp_["src_prompt"], _txt_s), token_span(mp_["dst_prompt"], _txt_d)       # raw task spans
    Ls = prepare_input(prompts=mp_["src_prompt"], tokenizer=mt)["input_ids"].shape[1]
    Ld = prepare_input(prompts=mp_["dst_prompt"], tokenizer=mt)["input_ids"].shape[1]
    cut = 0 if PP_V3_PIVOT else 1
    src_tok, dst_tok, src_w, dst_w, _, _ = align_kv_spans(
        mt, mp_["src_prompt"], mp_["dst_prompt"], (s0[0], Ls - cut), (d0[0], Ld - cut), mode="pad")
    assert len(src_w) == len(dst_w)
    d_ids = dst_tok["input_ids"][0].tolist()
    L = len(d_ids)
    pad_s, pad_d = src_tok["input_ids"].shape[1] - Ls, L - Ld
    lo = (max(i for i, t in enumerate(d_ids) if t == _im_end) if PP_I2_FROM == "im_end" else d0[1] + pad_d)
    seg = list(range(lo, L - 1))                                             # I2 window in these coordinates
    assert [d_ids[q] for q in seg] == [pp_dst_ids[q] for q in PP_SEG], "I2 window tokens changed under A1 padding"
    restore = None
    if PP_I2:
        if PP_I2_DONOR == "res":                                             # p_res aligned from the END
            assert _pp_res_knob == PP_RES_FRONTIER, "PP_RES_FRONTIER changed since the setup cell -- re-run it"
            restore = layer_outputs(pp_res_tok, [PP_L_RES - L + q for q in seg], PP_I2_LAYER_LIST)
        else:                                                                # clean DST, in these coordinates
            restore = layer_outputs(dst_tok, seg, PP_I2_LAYER_LIST)
    note = (f"A1: {max(pad_s, pad_d)} '\\n' pad token(s) before the {'SRC' if pad_s else 'DST'} window"
            if (pad_s or pad_d) else "A1: windows already equal length")
    return dict(src_tok=src_tok, dst_tok=dst_tok, src_w=src_w, dst_w=dst_w, seg=seg, restore=restore, note=note)


def pp_generate(tok, ctx_fn, first=None, max_new=None):
    """Greedy decode; ctx_fn(prompt_pass: bool) -> context manager for that forward. `first` forces the first token."""
    ids, am = tok["input_ids"], tok["attention_mask"]
    with torch.no_grad():
        with ctx_fn(True):
            out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
        pkv, pred0 = out.past_key_values, int(out.logits[0, -1].argmax())
        nxt = torch.tensor([[pred0 if first is None else first]], device=out.logits.device)
        gen = [int(nxt)]
        for _ in range(1, max_new or PP_MAX_NEW):
            am = torch.cat([am, torch.ones_like(nxt).to(am.device)], 1)
            with ctx_fn(False):
                out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv, use_cache=True)
            pkv, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
            if int(nxt) == mt.tokenizer.eos_token_id:
                break
            gen.append(int(nxt))
    return gen, pred0


def pp_variant(v):
    assert PP_CFG == (PP_SEED, PP_IDX, PP_SPAN, PP_I2_FROM), "knobs changed since the path-patching setup cell -- re-run it"
    assert repr(PP_MANUAL) == _pp_manual_used, "PP_MANUAL changed since the path-patching setup cell -- re-run it"
    cfg = PP_VARIANTS[v]
    until = cfg["prefill"] == "until"
    G = pp_geometry(cfg)
    src_tok, dst_tok, src_w, dst_w, seg = G["src_tok"], G["dst_tok"], G["src_w"], G["dst_w"], G["seg"]
    L = dst_tok["input_ids"].shape[1]
    piv_id = int(dst_tok["input_ids"][0, -1])
    assert not (until and max(dst_w) >= L - 1), "variant 4 cannot transplant the pivot (it is not in the prompt)"
    i1 = [(TASK_PATCH_HEADS, PP_I1_QUERY)] if PP_I1 else []
    i3 = [(GOAL_PATCH_HEADS, PP_I3_QUERY, {"blind_self": cfg["blind_self"], "v_scale": cfg.get("v_scale", PP_I3_V_SCALE)})] if PP_I3 else []
    if i1 or i3:
        hs = [h for r in i1 + i3 for h in r[0]]
        layers = sorted(heads_per_layer(hs, "k", layers={l for l, _ in hs}))
        k_c = cache_src_proj(src_tok, src_w, layers, "k")
        v_c = cache_src_proj(src_tok, src_w, layers, "v")

    def ctx(prompt_pass, full_prompt=False):
        """Interventions for one forward. Variant 4's prompt pass (prompt ends at `{var}`) has I1 + I2 only;
        I3 (and the boost) start at ` =`. `full_prompt` = a single forward over the prompt WITH ` =` (readout)."""
        s = ExitStack()
        i3_here = bool(i3) and not (until and prompt_pass and not full_prompt)
        routes = i1 + (i3 if i3_here else [])
        if routes:
            s.enter_context(headwise_kv_patch_routes(mt, routes, k_c, v_c, dst_w))
        if PP_I2 and prompt_pass:
            s.enter_context(residual_patch(G["restore"], seg, PP_I2_LAYER_LIST))
        if i3_here and PP_ALPHA != 1.0:
            s.enter_context(boost_head_ov(mt, GOAL_PATCH_HEADS, PP_ALPHA, positions=(-1,)))
        return s

    if until:
        tok = {"input_ids": dst_tok["input_ids"][:, :-1], "attention_mask": dst_tok["attention_mask"][:, :-1]}
        gen, pred0 = pp_generate(tok, ctx, first=piv_id if PP_V4_FORCE_PIVOT else None,
                                 max_new=PP_MAX_NEW + 1)   # +1: the ` =` step is not part of the generation
        text = mt.tokenizer.decode(gen[1:] if gen and gen[0] == piv_id else gen, skip_special_tokens=True)
        note = f"model's own token at `{{var}}`: {mt.tokenizer.decode([pred0])!r}" + (" (` =` forced)" if PP_V4_FORCE_PIVOT else "")
    else:
        gen, _ = pp_generate(dst_tok, ctx)
        text, note = mt.tokenizer.decode(gen, skip_special_tokens=True), G["note"]

    def iface(mt_, inputs, patches=None, model_kwargs=None):
        with ctx(True, full_prompt=True):
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))

    ro = goalscope_readout(f"   ↳ goalscope · variant {v}", dst_tok, L - 1, heads=READOUT_HEADS,
                           iface=iface)[0]["patched_gen"]
    ev = None
    if PP_ORACLE:
        from src.diff.eval import eval_kv_transfer
        ev = eval_kv_transfer(inst_pp, patched_gen=text, readouts=(pp_v_ro_src, pp_v_ro_dst, ro)).summary()
    return dict(variant=v, name=cfg["name"], window=f"{len(dst_w)} pos", gen=text, readout=ro, eval=ev, note=note)

In [92]:
# ── run the variants and compare (clean SRC / DST once) ──
pp_restore_v = pp_i2_cache() if PP_I2 else None   # I2 donor: p_res (a'b'e) or clean DST, per PP_I2_DONOR
pp_v_gen_src = greedy(pp_src_tok, max_new=PP_MAX_NEW)
pp_v_gen_dst = greedy(pp_dst_tok, max_new=PP_MAX_NEW)
pp_v_ro_src = goalscope_readout("   ↳ goalscope · clean SRC", pp_src_tok, PP_L_SRC - 1, heads=READOUT_HEADS)[0]["patched_gen"]
pp_v_ro_dst = goalscope_readout("   ↳ goalscope · clean DST", pp_dst_tok, PP_L_DST - 1, heads=READOUT_HEADS)[0]["patched_gen"]
if PP_I3 is False and any(v in (2, 3) for v in PP_COMPARE):
    print("note: PP_I3=False -> variant 2 equals 1 (no I3 heads to blind); variant 3 transplants span+progress for I1 only")
pp_rows = [pp_variant(v) for v in PP_COMPARE]

# C++ destination: show the whole generation (PP_MAX_NEW tokens); python: the first written line
PP_FULL_GEN = inst_pp.dst_lang == "cpp"
first = ((lambda g: g.strip()) if PP_FULL_GEN
         else (lambda g: next((ln for ln in g.split("\n") if ln.strip()), "").strip()))
t = Table(title=f"path-patching variants · instance {PP_IDX} · I1={PP_I1} I2={PP_I2} I3={PP_I3} · "
                f"c = {PP_C_DESC!r} · d = {PP_D_DESC!r}"
                + (f" · I2 donor e = {PP_E_DESC!r}" if PP_I2 and PP_I2_DONOR == "res" else " · I2 donor = clean DST"),
          show_lines=True)
for c_ in ("run", "transplant", f"generation ({PP_MAX_NEW} tok)" if PP_FULL_GEN else "first line", "goalscope @ ' ='", "eval"):
    t.add_column(c_, overflow="fold")
t.add_row(Text("clean SRC (c)", style="red1"), "—", first(pp_v_gen_src), pp_v_ro_src, "—")
t.add_row(Text("clean DST (d)", style="sea_green3"), "—", first(pp_v_gen_dst), pp_v_ro_dst, "—")
for r in pp_rows:
    t.add_row(Text(f"{r['variant']} · {r['name']}", style="salmon1"), r["window"],
              first(r["gen"]) + (f"\n[{r['note']}]" if r["note"] else ""), r["readout"], r["eval"] or "—")
Console(width=220).print(t)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · clean SRC  pos=147 ' =' => I'm checking if any element in the list is a palindrome by using a 
generator expression with `any()`.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · clean DST  pos=147 ' =' => I'm counting unique elements by converting the list to a set and 
measuring its length.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · variant 1  pos=147 ' =' => I'm checking if there are any duplicate elements in the list by 
comparing the length of the list with the length of the set of the list.

... (5 outputs left out)

   ↳ goalscope · variant 4  pos=147 ' =' => I'm checking if there are any duplicate elements in the list by 
comparing the length of the list with the length of the set of the list.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · variant 5  pos=147 ' =' => I'm checking the number of unique elements in the list by converting it
to a set and comparing its length to the original list's length.

                           path-patching variants · instance 11 · I1=False I2=False I3=True · c = 'whether any word is a palindrome' · d = 'the number of distinct words' · I2 donor = clean DST                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ run                                   ┃ transplant ┃ first line                                              ┃ goalscope @ ' ='                                        ┃ eval                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ clean SRC (c)                         │ —          │ any(w == w[::-1] for w in words)                        │ I'm checking if any element in the list is a palindrome │ —                                               │
│                                       │            │                                                         │ by using a generator expression with `any()`.           │                                                 │
├───────────────────────────────────────┼────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────┤
│ clean DST (d)                         │ —          │ len(set(words))                                         │ I'm counting unique elements by converting the list to  │ —                                               │
│                                       │            │                                                         │ a set and measuring its length.                         │                                                 │
├───────────────────────────────────────┼────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────┤
│ 1 · span k/v                          │ 30 pos     │ len(set(w for w in words if w == w[::-1]))              │ I'm checking if there are any duplicate elements in the │ causal: TRIGGERED (src) | readout: src→dst→dst  │
│                                       │            │                                                         │ list by comparing the length of the list with the       │                                                 │
│                                       │            │                                                         │ length of the set of the list.                          │                                                 │
├───────────────────────────────────────┼────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────┤
│ 2 · span k/v · I3 blind to self       │ 30 pos     │ len(set(w for w in words if w == w[::-1]))              │ I'm checking if there are any duplicate elements by     │ causal: TRIGGERED (src) | readout: src→dst→dst  │
│                                       │            │                                                         │ converting the list to a set and comparing its length   │                                                 │
│                                       │            │                                                         │ to the original list's length.                          │                                                 │
├───────────────────────────────────────┼────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼─────────────────────────────────────────────────┤
│ 3 · span + progress 